## Smoothie Sales Dashboard - Plotly & Dash Notes

A reference notebook covering the concepts, functions, and code used to build an interactive
sales dashboard with **Plotly Express** and **Dash**.

### 1. Installation & Imports

Install the two required libraries:

```bash
pip install plotly
pip install dash
```
<table>
<tr>
<th>Import</th>
<th>Purpose</th>
</tr>

<tr>
<td><b>pandas</b></td>
<td>Load and manipulate the sales data</td>
</tr>

<tr>
<td><b>plotly.express</b></td>
<td>Build interactive charts</td>
</tr>

<tr>
<td><b>dash</b> (<b>Dash, dcc, html, Input, Output</b>)</td>
<td>Build the web dashboard</td>
</tr>

</table>

In [9]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

### 2. Loading the Data

```bash
df = pd.read_csv("smoothie_sales.csv", parse_dates=["Date"])
```

**Why `parse_dates=["Date"]`?**

Without it, the `Date` column is read as plain text (`dtype: object`). With it, pandas converts
the column to a proper `datetime64` type, which is required for:

- Time-series plotting
- Date filtering

In [10]:
df = pd.read_csv("smoothie_sales.csv", parse_dates=["Date"])
df.head()

,Date,Smoothie,Sales
0,2025-06-01,Mango,200
1,2025-06-01,Banana,150
2,2025-06-01,Berry,120
3,2025-06-02,Mango,220
4,2025-06-02,Banana,180


In [11]:
df.dtypes

Date        datetime64[us]
Smoothie               str
Sales                int64
dtype: object

### 3. Building Dropdown Options - `.unique()` + List Comprehension

`.unique()` returns the distinct values in a column, dropping duplicates:

In [12]:
df["Smoothie"].unique()

<StringArray>
['Mango', 'Banana', 'Berry', 'Papaya']
Length: 4, dtype: str

This is used inside a **list comprehension** to build the dropdown's `options` list:

```bash
options = [
    {"label": smoothie, "value": smoothie}
    for smoothie in df["Smoothie"].unique()
]
```

Written as an ordinary loop, that is equivalent to:

```bash
options = []
for smoothie in df["Smoothie"].unique():
    options.append({"label": smoothie, "value": smoothie})
```

In [13]:
options = [
    {"label": smoothie, "value": smoothie}
    for smoothie in df["Smoothie"].unique()
]
options

[{'label': 'Mango', 'value': 'Mango'},
 {'label': 'Banana', 'value': 'Banana'},
 {'label': 'Berry', 'value': 'Berry'},
 {'label': 'Papaya', 'value': 'Papaya'}]

### 4. Creating the Dash Application

```bash
app = Dash(__name__)
```

`__name__` is a built-in Python variable identifying the current module - Dash uses it internally
to locate assets and templates. Think of `app` as *"the dashboard object"* everything else attaches to.

<h3>5. Defining the Layout</h3>

```bash
app.layout = html.Div([...])
```

<p>
The layout is the <b>frontend</b> - it answers <i>"what should the dashboard look like?"</i>.
Everything placed inside the outer list becomes part of the page.
</p>

<table>
  <tr>
    <th>Component</th>
    <th>Role</th>
  </tr>
  <tr>
    <td>html. (html.Div, html.H1, html.Label, html.P)</td>
    <td>Plain HTML elements</td>
  </tr>
  <tr>
    <td>dcc. (Dash Core Components)</td>
    <td>Interactive widgets, e.g. <b>dcc.Dropdown()</b>, <b>dcc.Graph()</b></td>
  </tr>
</table>

<ul>
  <li>
    <b>dcc.Dropdown(id="csv-smoothie-filter", options=options, value=..., clearable=False)</b>
    - the filter control.
  </li>
  <li>
    <b>dcc.Graph(id="csv-sales-over-time")</b>
    - an <b>empty placeholder</b>. Its <b>id</b> is what a callback later targets to fill it with a figure.
  </li>
</ul>

<p>So,</p>

<p><b>html</b> - Used to create HTML elements.</p>

<p><b>dcc</b> - It provides interactive components.</p>

<p>Example:</p>

<p><b>dcc.Dropdown()</b> - Creates a dropdown.</p>

<p><b>dcc.Graph()</b> - Creates an empty graph area or a place where a graph can be displayed.</p>

### 6. Callbacks - the Interactive Core

```bash
@app.callback(
    Output("csv-sales-over-time", "figure"),
    Input("csv-smoothie-filter", "value")
)
def update_csv_time_series(selected_smoothie):
    ...
```

<table>
  <tr>
    <th>Piece</th>
    <th>Meaning</th>
  </tr>
  <tr>
    <td><b>Input("csv-smoothie-filter", "value")</b></td>
    <td><i>"Watch the <b>value</b> property of the component with id <b>csv-smoothie-filter</b>."</i></td>
  </tr>
  <tr>
    <td><b>Output("csv-sales-over-time", "figure")</b></td>
    <td><i>"Write the function's return value into the <b>figure</b> property of the component with id <b>csv-sales-over-time</b>."</i></td>
  </tr>
</table>

<p>
Whenever the dropdown's <code>value</code> changes, Dash automatically re-runs the function and pushes the returned figure into the graph.
</p>

<p>Explanation : </p>
<p><b>Input</b> - Used in callbacks. It tells Dash :  </p>

<blockquote>
    "Watch this component for changes."
</blockquote>


<p><b>Output</b> -  It tells Dash :</p>

<blockquote>
    "Update this component when something changes."
</blockquote>


<p>For Input() There are two things:</p>

<p>
  'csv-smoothie-filter' = component ID
</p>

<p>
  'value' = property we want
</p>

<p>So:</p>

<p>
   Input('csv-smoothie-filter', 'value') means:
</p>

<blockquote>
    Watch the <code>value</code> of the dropdown whose ID is
    <code>csv-smoothie-filter</code>.
</blockquote>

<p>For Output() There are also two things:</p>

<p>
Output('csv-sales-over-time', 'figure') means : 
</p>

<blockquote>
    Put the result into the <code>figure</code> property of the graph with
    ID <code>csv-sales-over-time</code>.
</blockquote>

### 7. Filtering the DataFrame

```bash
filtered_df = df[df["Smoothie"] == selected_smoothie]
```

<p>
This is <b>boolean filtering</b>: pandas evaluates the condition row by row, producing a <b>True</b>/<b>False</b> mask, and keeps only the rows where it is <b>True</b>.
</p>

<table>
  <tr>
    <th>Row</th>
    <th><b>Smoothie</b></th>
    <th><b>== "Mango"</b></th>
  </tr>
  <tr>
    <td>0</td>
    <td>Mango</td>
    <td>True</td>
  </tr>
  <tr>
    <td>1</td>
    <td>Banana</td>
    <td>False</td>
  </tr>
  <tr>
    <td>2</td>
    <td>Mango</td>
    <td>True</td>
  </tr>
</table>

In [14]:
selected_smoothie = "Mango"
filtered_df = df[df["Smoothie"] == selected_smoothie]
filtered_df

,Date,Smoothie,Sales
0,2025-06-01,Mango,200
3,2025-06-02,Mango,220
6,2025-06-03,Mango,210


### 8. Aggregating Totals - `groupby().sum()`

```bash
total_sales = df.groupby("Smoothie", as_index=False)["Sales"].sum()
```

<ul>
  <li>
    <b>groupby("Smoothie")</b> groups all rows by smoothie name.
  </li>

  <li>
    <b>["Sales"]</b> selects only the <b>Sales</b> column <b>before</b> summing,
    so the sum is computed on that numeric column only. There is no need for the
    <b>numeric_only=True</b> argument here, since the non-numeric columns were
    already excluded by the column selection.
  </li>

  <li>
    <b>as_index=False</b> keeps <b>Smoothie</b> as a normal column instead
    of turning it into the index.
  </li>
</ul>

In [15]:
total_sales = df.groupby("Smoothie")["Sales"].sum()
total_sales

Smoothie
Banana    500
Berry     280
Mango     630
Papaya    150
Name: Sales, dtype: int64

In [16]:
total_sales = df.groupby("Smoothie", as_index=False)["Sales"].sum()
total_sales

,Smoothie,Sales
0,Banana,500
1,Berry,280
2,Mango,630
3,Papaya,150


### 9. Plotly Express - Quick Reference

<table>
  <tr>
    <th>Function</th>
    <th>Chart</th>
  </tr>
  <tr>
    <td><b>px.bar()</b></td>
    <td>Bar chart</td>
  </tr>
  <tr>
    <td><b>px.line()</b></td>
    <td>Line chart</td>
  </tr>
  <tr>
    <td><b>px.scatter()</b></td>
    <td>Scatter plot</td>
  </tr>
  <tr>
    <td><b>px.histogram()</b></td>
    <td>Histogram</td>
  </tr>
  <tr>
    <td><b>px.box()</b></td>
    <td>Box plot</td>
  </tr>
  <tr>
    <td><b>px.pie()</b></td>
    <td>Pie chart</td>
  </tr>
</table>

### 10. Running the Application

```bash
if __name__ == "__main__":
    app.run(debug=True)
```

- `app.run()` starts the Dash development server.
- `debug=True` enables hot-reloading on code changes and shows in-browser debugging info during development.

> This guard should not be run from inside a notebook — it starts a blocking web server. Use it
> only in a standalone `.py` script.

<p style="color: red;">Need to learn more and strengthen my understanding of the concepts....</p>